<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_10_custom_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-10: Custom Autograd Function

**Difficulty**: 🟡 Medium

**Objective**: Write custom forward AND backward passes. Needed for Flash Attention, custom kernels, and straight-through estimators.

In [1]:
import torch
from torch.autograd import Function

## Task 1: Custom ReLU with Explicit Backward

In [3]:
class MyReLU(Function):
    @staticmethod
    def forward(ctx, x):
        # TODO: Compute ReLU, save what's needed for backward
        # ctx.save_for_backward(...)  — save the input for the backward pass
        x = x * (x > 0)
        ctx.save_for_backward(x)
        return x

    @staticmethod
    def backward(ctx, grad_output):
        # TODO: Compute gradient of ReLU
        # d/dx ReLU(x) = 1 if x > 0, else 0
        x, = ctx.saved_tensors
        grad_output = grad_output * (x > 0)
        return grad_output

# Test
x = torch.randn(5, requires_grad=True)
y = MyReLU.apply(x)
y.sum().backward()

# Compare against PyTorch's ReLU
x2 = x.detach().clone().requires_grad_(True)
y2 = torch.relu(x2)
y2.sum().backward()

assert torch.allclose(x.grad, x2.grad)
print('✅ Task 1 passed!')

✅ Task 1 passed!


## Task 2: Straight-Through Estimator

The STE: forward does hard threshold (non-differentiable), backward pretends it was identity. Used in quantization.

In [4]:
class StraightThrough(Function):
    @staticmethod
    def forward(ctx, x):
        # TODO: Return sign(x): -1, 0, or 1 (hard threshold)
        sign = x.sign()
        return sign

    @staticmethod
    def backward(ctx, grad_output):
        # TODO: Pass gradient through unchanged (identity)
        return grad_output

x = torch.randn(10, requires_grad=True)
y = StraightThrough.apply(x)
# Forward should be binarized
assert set(y.unique().tolist()).issubset({-1.0, 0.0, 1.0})
# Backward should pass through
y.sum().backward()
assert torch.allclose(x.grad, torch.ones_like(x))
print('✅ Task 2 passed!')

✅ Task 2 passed!
